## (01) Install & Import Dependencies

In [1]:
import os, re, time, random, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from collections import defaultdict

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

import nltk
from nltk.tokenize import word_tokenize
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

import gensim
from gensim.models import Word2Vec, FastText, KeyedVectors
import gensim.downloader as gensim_api

from transformers import BertTokenizerFast, BertModel, AutoTokenizer
from torchcrf import CRF
from seqeval.metrics import f1_score, classification_report

warnings.filterwarnings('ignore')

# ─── Reproducibility ─────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')   # ← Apple GPU acceleration
else:
    DEVICE = torch.device('cpu')

print(f'Using device: {DEVICE}')

Using device: mps


## (02) Load BC5CDR Dataset

In [6]:
# ── Set your paths here ──────────────────────────────────────────────────────
TRAIN_FILE = 'train.txt'
VAL_FILE   = 'val.txt'
TEST_FILE  = 'test.txt'

# ─── Parser ───────────────────────────────────────────────────────────────────
def read_conll(filepath):
    """Read CoNLL-style file. Returns list of (tokens, tags) tuples.
    Handles both TAB and SPACE separators, and skips doc-start lines."""
    sentences, tokens, tags = [], [], []
    with open(filepath, encoding='utf-8') as f:
        for line in f:
            line = line.rstrip('\n')
            if line.startswith('-DOCSTART-') or line.startswith('#'):
                continue
            if line.strip() == '':
                if tokens:
                    sentences.append((tokens, tags))
                    tokens, tags = [], []
            else:
                parts = line.split('\t') if '\t' in line else line.split()
                if len(parts) >= 2:
                    tokens.append(parts[0])
                    tags.append(parts[-1])   # last column = NER tag
    if tokens:
        sentences.append((tokens, tags))
    return sentences

train_data = read_conll(TRAIN_FILE)
val_data   = read_conll(VAL_FILE)
test_data  = read_conll(TEST_FILE)

print(f'Train sentences : {len(train_data)}')
print(f'Val   sentences : {len(val_data)}')
print(f'Test  sentences : {len(test_data)}')
print('\nSample sentence:')
print(train_data[0])

Train sentences : 4560
Val   sentences : 4581
Test  sentences : 4797

Sample sentence:
(['Selegiline', '-', 'induced', 'postural', 'hypotension', 'in', 'Parkinson', "'", 's', 'disease', ':', 'a', 'longitudinal', 'study', 'on', 'the', 'effects', 'of', 'drug', 'withdrawal', '.'], ['O', 'O', 'O', 'B-Disease', 'I-Disease', 'O', 'B-Disease', 'I-Disease', 'I-Disease', 'I-Disease', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O'])


In [3]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# Must be before ANY other TF import
import tensorflow as tf
tf.compat.v1.enable_eager_execution()

import tensorflow_hub as hub

print(f'TF version: {tf.__version__}')
print(f'Eager mode: {tf.executing_eagerly()}')  # must print True

print('Loading ELMo...')
elmo_tf = hub.load('https://tfhub.dev/google/elmo/3')
ELMO_DIM = 1024
print(f'ELMo loaded.')

TF version: 2.21.0
Eager mode: True
Loading ELMo...
ELMo loaded.


In [4]:
# Quick test — should print a tensor of shape (1, 5, 1024)
test_tokens = ['patients', 'with', 'diabetes', 'mellitus', 'treated']
result = elmo_tf.signatures['tokens'](
    tokens=tf.constant([test_tokens]),
    sequence_len=tf.constant([len(test_tokens)])
)
emb = result['elmo'].numpy()
print(f'Shape: {emb.shape}')   # should be (1, 5, 1024)
print('ELMo is working correctly!')

Shape: (1, 5, 1024)
ELMo is working correctly!
